# Experiment 4: Pooling Strategy Ablation

Compares GAP vs max vs top-k pooling strategies for extracting per-layer
activation summaries from the frozen backbone.

For each strategy, evaluates Criterion 1 (R²) and Criterion 2 (Spearman ρ).

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.metrics import geometric_consistency

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Config and checkpoint paths for each pooling strategy
# TODO: Update checkpoint paths once ablation experiments are trained
pool_configs = {
    'gap':  CONFIG_DIR + '/phase1.yaml',
    'max':  CONFIG_DIR + '/ablations/max_pool.yaml',
    'topk': CONFIG_DIR + '/ablations/top_k_pool.yaml',
}
pool_checkpoints = {
    'gap':  CKPT_DIR + '/phase1/best.pt',
    'max':  CKPT_DIR + '/ablations/max_pool/best.pt',
    'topk': CKPT_DIR + '/ablations/top_k_pool/best.pt',
}

In [ ]:
# Evaluate each pooling strategy
results = {}
MAX_PAIRS = 50_000

for pool_mode, config_path in pool_configs.items():
    print(f'\n=== {pool_mode.upper()} ===')
    with open(config_path) as f:
        config = yaml.safe_load(f)
    config['data']['data_dir'] = DATA_DIR

    ckpt_path = pool_checkpoints[pool_mode]
    if not os.path.exists(ckpt_path):
        print(f'  Checkpoint not found: {ckpt_path}')
        continue

    backbone = FrozenBackbone(
        arch=config['model']['arch'],
        num_classes=config['model']['num_classes'],
        pretrained=config['model']['pretrained'],
        pool_mode=config['model'].get('pool_mode', 'gap')
    ).to(device)

    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        projection_dim=config['model']['meta_encoder']['projection_dim'],
        n_heads=config['model']['meta_encoder']['n_heads'],
        n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
        dropout=config['model']['meta_encoder']['dropout']
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    meta_encoder.eval()

    _, val_loader = get_standard_loaders(
        data_dir=DATA_DIR,
        batch_size=config['data'].get('batch_size', 256),
        num_workers=0,
        augment=False,
        download=False,
    )

    analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
    data     = analyzer.collect_representations(max_samples=2000)

    N = data['z_list'][0].shape[0]
    idx_a, idx_b = torch.triu_indices(N, N, offset=1)
    if idx_a.shape[0] > MAX_PAIRS:
        perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
        idx_a, idx_b = idx_a[perm], idx_b[perm]

    true_sims    = CircuitAnalyzer.compute_pair_profiles(data['trajectories'], idx_a, idx_b)
    true_sims_np = true_sims.numpy()

    L = len(data['z_list'])
    z_sims_np = np.zeros((idx_a.shape[0], L))
    for l in range(L):
        z_sims_np[:, l] = (
            data['z_list'][l][idx_a].numpy() * data['z_list'][l][idx_b].numpy()
        ).sum(axis=1)

    gc = geometric_consistency(z_sims_np, true_sims_np, L)
    results[pool_mode] = gc
    print(f"  Mean Spearman \u03c1: {gc['mean_rho']:.4f}  (passes: {gc['passes']})")

In [ ]:
# Bar chart comparison
if results:
    modes  = list(results.keys())
    rhos   = [results[m]['mean_rho'] for m in modes]
    colors = ['steelblue', 'coral', 'mediumseagreen']

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(modes, rhos, color=colors[:len(modes)])
    ax.axhline(y=0.65, color='r', linestyle='--', label='Target (0.65)')
    ax.set_ylabel('Mean Spearman \u03c1')
    ax.set_title('Pooling Strategy Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()